# Validación del modelo en condiciones reales (nevera)

Este notebook mide el rendimiento del detector entrenado sobre fotografías reales de nevera/despensa, comparando las detecciones de YOLOv8 con un *ground truth* anotado manualmente (precisión, recall y F1 por imagen).

**Nota metodológica (pendiente antes de citar estos resultados en el TFM):** la primera ejecución de este notebook se hizo con solo 2 imágenes de banco de imágenes (Unsplash), no con fotografías reales de una nevera. Ese resultado (25% de precisión) no es representativo y **no debe usarse como validación final**. Antes de redactar el Capítulo 5 con estos datos:
1. Sustituir las imágenes de ejemplo por un conjunto propio de fotografías reales de nevera/despensa (se recomienda un mínimo de 15-20 imágenes con distintas condiciones de luz y desorden).
2. Confirmar que `MODEL_PATH` apunta al modelo fine-tuned entrenado en el notebook 04 (o al modelo final documentado en el Capítulo 4), no a un YOLOv8 genérico preentrenado en COCO.
3. Ejecutar el **Modo B** (por lotes) para obtener un resultado reproducible y no dependiente de teclear el ground truth a mano en cada sesión.

In [ ]:
!pip install ultralytics --quiet

In [ ]:
import os
import shutil
from pathlib import Path

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ── Configuración ────────────────────────────────────────────────────────────
# IMPORTANTE: apunta esto a los pesos del modelo entrenado y evaluado en el
# notebook 04 (o al modelo final que se documente en el Capítulo 4 como
# arquitectura de producción). No usar un modelo genérico preentrenado en
# COCO: sus clases (80 categorías genéricas tipo "apple", "banana", "bottle")
# no corresponden a las categorías de alimentos entrenadas para este TFM.
MODEL_PATH       = "results/yolov8_uec256/weights/best.pt"
CONFIANZA_MINIMA = 0.30

TRADUCCIONES = {
    "rice": "arroz", "sushi": "sushi", "pizza": "pizza",
    "hamburger": "hamburguesa", "sandwiches": "sándwich",
    "spaghetti": "espagueti", "toast": "tostada",
    "croissant": "croissant", "fried rice": "arroz frito",
    "ramen noodle": "ramen", "beef curry": "curry de ternera",
    "sauteed vegetables": "verduras salteadas",
    # Ampliar según las 30 categorías de UEC-256 usadas en el notebook 04
    # y las clases reales que devuelva `modelo.names` al cargarlo.
}

print("═" * 60)
print("  VALIDACIÓN — YOLOv8 en imágenes de nevera")
print("  TFM Sistema Inteligente de Gestión Alimentaria")
print("═" * 60)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"No se encuentra el modelo en '{MODEL_PATH}'. "
        "Actualiza MODEL_PATH con la ruta a tus pesos entrenados (best.pt)."
    )

modelo = YOLO(MODEL_PATH)
print(f"✅ Modelo cargado desde: {MODEL_PATH}")
print(f"   Clases del modelo: {list(modelo.names.values())}")


def detectar_y_evaluar(nombre, ruta_imagen, ground_truth, mostrar=True):
    """
    Ejecuta la detección sobre una imagen y calcula precisión, recall y F1
    frente a una lista de ingredientes reales (ground truth).

    Args:
        nombre:        nombre identificativo de la imagen (para el reporte)
        ruta_imagen:   ruta al archivo de imagen
        ground_truth:  lista de nombres de ingredientes presentes realmente
                        en la imagen (en español o en la clase del modelo)
        mostrar:       si True, dibuja la imagen con las detecciones
    Returns:
        dict con la fila de resultados para esta imagen
    """
    yolo_result = modelo(ruta_imagen, conf=CONFIANZA_MINIMA, iou=0.5, verbose=False)

    detecciones = []
    for r in yolo_result:
        for box in r.boxes:
            etiqueta = modelo.names[int(box.cls)]
            confianza = float(box.conf)
            detecciones.append((etiqueta, round(confianza, 3)))
    detecciones.sort(key=lambda x: x[1], reverse=True)

    if mostrar:
        imagen_rgb = cv2.cvtColor(cv2.imread(ruta_imagen), cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f"{nombre}", fontsize=13, fontweight="bold")

        axes[0].imshow(imagen_rgb)
        axes[0].set_title("Imagen original", fontsize=11)
        axes[0].axis("off")

        axes[1].axis("off")
        if detecciones:
            texto = "INGREDIENTES DETECTADOS POR YOLO:\n\n"
            for i, (etq, conf) in enumerate(detecciones, 1):
                trad = TRADUCCIONES.get(etq, etq)
                barra = "█" * int(conf * 20)
                texto += f"{i}. {trad}  ({etq})\n   {conf:.0%}  {barra}\n\n"
        else:
            texto = (
                "YOLO no detectó ningún objeto\ncon confianza ≥ "
                f"{CONFIANZA_MINIMA:.0%}.\n\nPosibles causas:\n"
                "• Imagen con poca luz\n• Alimentos dentro de envases\n"
                "• Ángulo o distancia difícil"
            )
        axes[1].text(
            0.05, 0.95, texto, transform=axes[1].transAxes,
            fontsize=10, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="#f0f4f8", alpha=0.9)
        )
        plt.tight_layout()
        plt.show()
        plt.close()

    detectados_es = set()
    for etq, _ in detecciones:
        detectados_es.add(TRADUCCIONES.get(etq, etq).lower())
        detectados_es.add(etq.lower())

    gt_set = {g.strip().lower() for g in ground_truth if g.strip()}
    vp = len(gt_set & detectados_es)
    fp = len(detectados_es - gt_set)
    fn = len(gt_set - detectados_es)

    precision = vp / (vp + fp) if (vp + fp) > 0 else 0.0
    recall    = vp / (vp + fn) if (vp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    print(f"  {nombre}: {len(detecciones)} detectados por YOLO | "
          f"Precisión: {precision:.0%}  Recall: {recall:.0%}  F1: {f1:.0%}")

    return {
        "imagen":                  nombre,
        "num_detectados_yolo":     len(detecciones),
        "num_reales_ground_truth": len(gt_set),
        "detectados_yolo":         "; ".join(
            [f"{TRADUCCIONES.get(e,e)} ({c:.0%})" for e, c in detecciones]
        ),
        "ground_truth":            "; ".join(sorted(gt_set)),
        "verdaderos_positivos":    vp,
        "falsos_positivos":        fp,
        "falsos_negativos":        fn,
        "precision":               round(precision, 3),
        "recall":                  round(recall, 3),
        "f1_score":                round(f1, 3),
    }


def resumen_y_grafico(resultados, sufijo="resultados"):
    """Genera el resumen agregado, el gráfico de métricas y guarda el CSV."""
    if not resultados:
        print("⚠️  No hay resultados que resumir.")
        return None

    df = pd.DataFrame(resultados)
    print("\n" + "═" * 60)
    print("  RESUMEN FINAL")
    print("═" * 60)
    print(f"  Imágenes analizadas:   {len(df)}")
    print(f"  Precisión media:       {df['precision'].mean():.1%}")
    print(f"  Recall medio:          {df['recall'].mean():.1%}")
    print(f"  F1-score medio:        {df['f1_score'].mean():.1%}")
    print(f"  Total aciertos (VP):   {int(df['verdaderos_positivos'].sum())}")
    print(f"  Total errores (FP):    {int(df['falsos_positivos'].sum())}")
    print(f"  Total omisiones (FN):  {int(df['falsos_negativos'].sum())}")
    print("═" * 60)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Validación — Resultados finales", fontsize=14, fontweight="bold")

    x = range(len(df))
    ax = axes[0]
    ax.plot(x, df["precision"], "o-", label="Precisión", color="#2196F3", linewidth=2)
    ax.plot(x, df["recall"],    "s-", label="Recall",    color="#4CAF50", linewidth=2)
    ax.plot(x, df["f1_score"],  "^-", label="F1-score",  color="#FF9800", linewidth=2)
    ax.set_xlabel("Imagen", fontsize=11)
    ax.set_ylabel("Valor (0–1)", fontsize=11)
    ax.set_title("Métricas por imagen", fontsize=12)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df["imagen"].tolist(), rotation=45, ha="right", fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    medias = {
        "Precisión": df["precision"].mean(),
        "Recall":    df["recall"].mean(),
        "F1-score":  df["f1_score"].mean(),
    }
    bars = ax2.bar(medias.keys(), medias.values(),
                    color=["#2196F3", "#4CAF50", "#FF9800"], width=0.5)
    for bar, val in zip(bars, medias.values()):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                  f"{val:.1%}", ha="center", fontsize=13, fontweight="bold")
    ax2.set_ylim(0, 1.15)
    ax2.set_title("Métricas medias globales", fontsize=12)
    ax2.set_ylabel("Valor (0–1)", fontsize=11)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(f"grafico_{sufijo}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    ruta_csv = f"{sufijo}.csv"
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    print(f"\n💾 Guardado: {ruta_csv} y grafico_{sufijo}.png")
    return df


## Modo A — Validación interactiva

Subida manual de fotos en Colab, anotando el ground truth sobre la marcha. Útil para pruebas exploratorias rápidas; al depender de `input()`, no se puede ejecutar con "Run all" sin supervisión.

In [ ]:
# ── Modo A: validación interactiva (subir fotos y anotar en el momento) ──────
# Pensado para Google Colab. Sube varias fotos de nevera reales y, para cada
# una, escribe a mano qué ingredientes contiene realmente.
from google.colab import files

CARPETA_TEMP = "/content/imagenes_nevera"
os.makedirs(CARPETA_TEMP, exist_ok=True)

print("📂 Sube tus fotos de nevera (varias a la vez con Ctrl/Cmd+clic)\n")
archivos_subidos = files.upload()

rutas_imagenes = []
for nombre, contenido in archivos_subidos.items():
    ruta = os.path.join(CARPETA_TEMP, nombre)
    with open(ruta, "wb") as f:
        f.write(contenido)
    rutas_imagenes.append((nombre, ruta))

print(f"\n✅ {len(rutas_imagenes)} imagen(es) subida(s).\n")

resultados_interactivo = []
for idx, (nombre, ruta) in enumerate(rutas_imagenes, 1):
    print(f"\n[{idx}/{len(rutas_imagenes)}] {nombre}")

    fila_preview = detectar_y_evaluar(nombre, ruta, ground_truth=[], mostrar=True)
    detectados_preview = fila_preview["detectados_yolo"]
    print(f"  YOLO detectó: {detectados_preview or '(nada)'}")

    print("  ¿Qué ingredientes hay REALMENTE en esta imagen? (separados por comas)")
    print("  Escribe SALTAR si la imagen no sirve.")
    respuesta = input("  → ").strip()

    if respuesta.upper() == "SALTAR":
        print("  ⏭️  Imagen saltada.")
        continue

    ground_truth = [x.strip() for x in respuesta.split(",") if x.strip()]
    fila = detectar_y_evaluar(nombre, ruta, ground_truth, mostrar=False)
    resultados_interactivo.append(fila)

df_interactivo = resumen_y_grafico(resultados_interactivo, sufijo="validacion_interactiva")


## Modo B — Validación por lotes (reproducible)

Lee las imágenes y el ground truth desde disco, sin intervención manual durante la ejecución. Es el modo recomendado para el resultado que se documente en el TFM, ya que se puede volver a ejecutar de forma idéntica cuantas veces sea necesario.

In [ ]:
# ── Modo B: validación por lotes (carpeta de imágenes + CSV de ground truth) ─
# Alternativa reproducible al Modo A: no requiere teclear nada durante la
# ejecución, por lo que el notebook se puede correr de principio a fin
# ("Run all") de forma automática. Útil para volver a evaluar el modelo
# cuantas veces haga falta según se añadan más fotos.
#
# Estructura esperada:
#   IMAGENES_DIR/
#     foto_01.jpg
#     foto_02.jpg
#     ...
#   GROUND_TRUTH_CSV con columnas:
#     imagen, ingredientes_reales
#     foto_01.jpg, "manzana; leche; zanahoria"
#     foto_02.jpg, "tomate; pimiento"

IMAGENES_DIR      = "imagenes_nevera_reales"
GROUND_TRUTH_CSV  = "ground_truth_nevera.csv"

if not os.path.exists(GROUND_TRUTH_CSV):
    print(f"⚠️  No se encuentra '{GROUND_TRUTH_CSV}'. Este modo requiere un CSV "
          "con columnas 'imagen' e 'ingredientes_reales' (separados por ';'). "
          "Créalo una vez tengas un conjunto de fotos reales anotado y vuelve "
          "a ejecutar esta celda.")
else:
    gt_df = pd.read_csv(GROUND_TRUTH_CSV)
    resultados_lote = []

    for _, fila_gt in gt_df.iterrows():
        nombre = fila_gt["imagen"]
        ruta = os.path.join(IMAGENES_DIR, nombre)

        if not os.path.exists(ruta):
            print(f"  ⚠️  No encontrada: {ruta} — se omite.")
            continue

        ground_truth = [g for g in str(fila_gt["ingredientes_reales"]).split(";")]
        fila = detectar_y_evaluar(nombre, ruta, ground_truth, mostrar=False)
        resultados_lote.append(fila)

    df_lote = resumen_y_grafico(resultados_lote, sufijo="validacion_lote")
